In [1]:
import os
os.chdir('/home/smallyan/eval_agent')
print(f"Working directory: {os.getcwd()}")

Working directory: /home/smallyan/eval_agent


# Code Evaluation: Modular Addition Circuit Analysis

## Evaluation Mode: No-Execution Code Evaluation

This notebook evaluates the code in `/net/scratch2/smallyan/modular_addition_claude_2025-12-26_01-20-45` by static inspection only.

### Evaluation Criteria

For each code block, we evaluate:
1. **Runnable (Y/N)**: Is the code syntactically valid and able to execute without obvious runtime errors?
2. **Correct-Implementation (Y/N)**: Does the code correctly implement the intended logic as described in the plan/codewalk?
3. **Redundant (Y/N)**: Does the block duplicate another block's computation?
4. **Irrelevant (Y/N)**: Does the block contribute to the project goal?

## Project Goal Summary

From the **Plan** and **Codewalk** files:

**Goal**: Reverse-engineer the modular addition circuit in a 1-layer transformer trained on `(a + b) mod p` to understand how the model learns to use Discrete Fourier Transform (DFT) representations for arithmetic.

**Phases**:
1. Setup and Training - Train a 1-layer transformer
2. Embedding Analysis - FFT analysis on embedding vectors
3. Attention Head Analysis - Analyze attention patterns
4. MLP Analysis - Analyze MLP neuron activations
5. Ablation Studies - Ablate components and measure impact
6. Circuit Documentation - Compile findings

In [2]:
# Define the evaluation data structure
import pandas as pd
import json

# Analysis of each cell in the notebook
# Based on static inspection of the code

evaluation_data = [
    # Cell 0: Setup - change directory
    {"Cell": "Cell 0", "Type": "code", "Description": "Change working directory and print",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": ""},
    # Cell 1: Markdown - intro
    {"Cell": "Cell 1", "Type": "markdown", "Description": "Introduction and hypothesis",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Documentation cell"},
    # Cell 2: Check environment and GPU
    {"Cell": "Cell 2", "Type": "code", "Description": "Import torch, numpy, matplotlib; check GPU",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Standard setup code"},
    # Cell 3: Check transformer_lens availability
    {"Cell": "Cell 3", "Type": "code", "Description": "Try import transformer_lens, import other libs",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Handles import error gracefully"},
    # Cell 4: Import transformer_lens again
    {"Cell": "Cell 4", "Type": "code", "Description": "Import transformer_lens again, set TL_AVAILABLE",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "Y", "Irrelevant": "N",
     "Notes": "Redundant: Re-imports same libraries as Cell 3. Correction cell after version check failed."},
    # Cell 5: Markdown - Phase 1 setup
    {"Cell": "Cell 5", "Type": "markdown", "Description": "Phase 1 dataset creation intro",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Documentation cell"},
    # Cell 6: Create dataset
    {"Cell": "Cell 6", "Type": "code", "Description": "Create modular addition dataset p=113",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Creates all pairs correctly: (a+b) mod p"},
    # Cell 7: PyTorch dataset class
    {"Cell": "Cell 7", "Type": "code", "Description": "Define ModularAdditionDataset class, split train/test",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct implementation of Dataset class"},
    # Cell 8: First model definition
    {"Cell": "Cell 8", "Type": "code", "Description": "Define ModularAdditionTransformer class (v1)",
     "Runnable": "Y", "Correct_Implementation": "N", "Redundant": "N", "Irrelevant": "N",
     "Notes": "Incorrect einsum: 'bsd,ndk->bsnk' uses 'k' for two different dimensions. Fixed in Cell 10."},
    # Cell 9: Training function
    {"Cell": "Cell 9", "Type": "code", "Description": "Define train_epoch, evaluate functions, start training",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Training code is correct."},
    # Cell 10: Fixed model definition
    {"Cell": "Cell 10", "Type": "code", "Description": "Fixed ModularAdditionTransformer class (v2)",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N",
     "Notes": "Correctly fixes einsum operations: 'bsd,ndh->bsnh', 'bqnh,bknh->bnqk', etc."},
    # Cell 11: Continue training
    {"Cell": "Cell 11", "Type": "code", "Description": "Train the model with scheduler",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Uses corrected model from Cell 10"},
    # Cell 12: Continue training for grokking
    {"Cell": "Cell 12", "Type": "code", "Description": "Continue training for grokking effect",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correctly continues training"},
    # Cell 13: More training
    {"Cell": "Cell 13", "Type": "code", "Description": "Continue training more",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Additional training iterations"},
    # Cell 14: Plot training curves
    {"Cell": "Cell 14", "Type": "code", "Description": "Plot training curves, save figure",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Visualization of grokking"},
    # Cell 15: Markdown - Phase 2
    {"Cell": "Cell 15", "Type": "markdown", "Description": "Phase 2 Fourier analysis intro",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Documentation cell"},
    # Cell 16: Extract embeddings and FFT
    {"Cell": "Cell 16", "Type": "code", "Description": "Extract embedding weights, compute FFT, plot",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct FFT analysis"},
    # Cell 17: Find dominant frequencies
    {"Cell": "Cell 17", "Type": "code", "Description": "Find dominant frequencies, define compute_fourier_basis",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct Fourier basis computation"},
    # Cell 18: Project onto Fourier basis
    {"Cell": "Cell 18", "Type": "code", "Description": "Project embeddings onto Fourier basis",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct projection"},
    # Cell 19: Visualize embeddings
    {"Cell": "Cell 19", "Type": "code", "Description": "Visualize example embeddings",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Visualization"},
    # Cell 20: Find dimensions with Fourier content
    {"Cell": "Cell 20", "Type": "code", "Description": "Find dimensions with strongest Fourier features",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct correlation analysis"},
    # Cell 21: Find Fourier pairs
    {"Cell": "Cell 21", "Type": "code", "Description": "Define find_fourier_pairs, find cos/sin pairs",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct implementation"},
    # Cell 22: Analyze unembedding
    {"Cell": "Cell 22", "Type": "code", "Description": "Analyze unembedding Fourier structure",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Uses find_fourier_pairs"},
    # Cell 23: Visualize unembedding Fourier
    {"Cell": "Cell 23", "Type": "code", "Description": "Visualize unembedding Fourier structure",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct visualization"},
    # Cell 24: Markdown - Phase 3
    {"Cell": "Cell 24", "Type": "markdown", "Description": "Phase 3 attention analysis intro",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Documentation cell"},
    # Cell 25: Attention patterns
    {"Cell": "Cell 25", "Type": "code", "Description": "Analyze attention patterns, create test batch",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct attention analysis"},
    # Cell 26: Head output analysis
    {"Cell": "Cell 26", "Type": "code", "Description": "Analyze head outputs correlation with Fourier features",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct analysis"},
    # Cell 27: MLP output analysis
    {"Cell": "Cell 27", "Type": "code", "Description": "Analyze MLP output Fourier correlation",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct MLP analysis"},
    # Cell 28: Summary of Fourier correlations
    {"Cell": "Cell 28", "Type": "code", "Description": "Create summary table of Fourier correlations",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Aggregates analysis results"},
    # Cell 29: MLP hidden analysis
    {"Cell": "Cell 29", "Type": "code", "Description": "Analyze MLP hidden neuron activations",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct analysis"},
    # Cell 30: MLP neuron Fourier correlation
    {"Cell": "Cell 30", "Type": "code", "Description": "Analyze MLP neuron Fourier correlations",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct correlation analysis"},
    # Cell 31: Embedding Fourier analysis by position
    {"Cell": "Cell 31", "Type": "code", "Description": "Analyze embedding Fourier at each position",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Position-wise analysis"},
    # Cell 32: Circuit mechanism summary
    {"Cell": "Cell 32", "Type": "code", "Description": "Print circuit mechanism summary",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Summary output"},
    # Cell 33: Markdown - Phase 4
    {"Cell": "Cell 33", "Type": "markdown", "Description": "Phase 4 ablation studies intro",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Documentation cell"},
    # Cell 34: Ablation function and tests
    {"Cell": "Cell 34", "Type": "code", "Description": "Define evaluate_with_ablation, run ablations",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct ablation implementation"},
    # Cell 35: Visualize ablation results
    {"Cell": "Cell 35", "Type": "code", "Description": "Visualize ablation results",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct visualization"},
    # Cell 36: Pair ablations
    {"Cell": "Cell 36", "Type": "code", "Description": "Ablate pairs of heads",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct pair ablation"},
    # Cell 37: Save circuit
    {"Cell": "Cell 37", "Type": "code", "Description": "Create and save circuit JSON",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct output generation"},
    # Cell 38: Markdown - Final summary
    {"Cell": "Cell 38", "Type": "markdown", "Description": "Final summary markdown",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Documentation cell"},
    # Cell 39: Save model
    {"Cell": "Cell 39", "Type": "code", "Description": "Save model checkpoint",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct model saving"},
    # Cell 40: Summary visualization
    {"Cell": "Cell 40", "Type": "code", "Description": "Create summary visualization figure",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Correct visualization"},
    # Cell 41: Final summary print
    {"Cell": "Cell 41", "Type": "code", "Description": "Print final summary",
     "Runnable": "Y", "Correct_Implementation": "Y", "Redundant": "N", "Irrelevant": "N", "Notes": "Summary output"},
]

# Create DataFrame
df = pd.DataFrame(evaluation_data)
print(f"Total cells evaluated: {len(df)}")
print(f"Code cells: {len(df[df['Type'] == 'code'])}")
print(f"Markdown cells: {len(df[df['Type'] == 'markdown'])}")

Total cells evaluated: 42
Code cells: 36
Markdown cells: 6


## Block-Level Evaluation Table

In [3]:
# Display the evaluation table
from IPython.display import display

# Format the table nicely
table_df = df[['Cell', 'Type', 'Description', 'Runnable', 'Correct_Implementation', 'Redundant', 'Irrelevant', 'Notes']]
table_df.columns = ['Cell', 'Type', 'Description', 'Runnable', 'Correct-Impl', 'Redundant', 'Irrelevant', 'Notes']

# Display as styled table
display(table_df)

,Cell,Type,Description,Runnable,Correct-Impl,Redundant,Irrelevant,Notes
0,Cell 0,code,Change working directory and print,Y,Y,N,N,
1,Cell 1,markdown,Introduction and hypothesis,Y,Y,N,N,Documentation cell
2,Cell 2,code,"Import torch, numpy, matplotlib; check GPU",Y,Y,N,N,Standard setup code
3,Cell 3,code,"Try import transformer_lens, import other libs",Y,Y,N,N,Handles import error gracefully
4,Cell 4,code,"Import transformer_lens again, set TL_AVAILABLE",Y,Y,Y,N,Redundant: Re-imports same libraries as Cell 3...
5,Cell 5,markdown,Phase 1 dataset creation intro,Y,Y,N,N,Documentation cell
6,Cell 6,code,Create modular addition dataset p=113,Y,Y,N,N,Creates all pairs correctly: (a+b) mod p
7,Cell 7,code,"Define ModularAdditionDataset class, split tra...",Y,Y,N,N,Correct implementation of Dataset class
8,Cell 8,code,Define ModularAdditionTransformer class (v1),Y,N,N,N,"Incorrect einsum: 'bsd,ndk->bsnk' uses 'k' for..."
9,Cell 9,code,"Define train_epoch, evaluate functions, start ...",Y,Y,N,N,Training code is correct.


## Detailed Error Notes

### Cell 4 - Redundant
**Issue**: Re-imports the same libraries (`transformer_lens`, `nn`, `F`, `Dataset`, `DataLoader`, `tqdm`, `json`) that were already imported in Cell 3. This appears to be a correction cell after Cell 3's try block may have failed to print the version (the `__version__` attribute doesn't exist), but the imports themselves are duplicated.

### Cell 8 - Incorrect Implementation
**Issue**: The einsum operations have incorrect index handling:
- `'bsd,ndk->bsnk'` uses 'k' for two different dimensions (d_head output and the contracted index)
- `'bqnk,bknk->bnqk'` has 'k' appearing three times with different meanings
- `'bnqk,bknk->bqnk'` similarly has index confusion

This version of the model is corrected in Cell 10, where the einsums are fixed to:
- `'bsd,ndh->bsnh'` (correct: b=batch, s=seq, d=d_model, n=n_heads, h=d_head)
- `'bqnh,bknh->bnqk'` (correct: q=query_seq, k=key_seq as separate indices)

## Quantitative Metrics

In [4]:
# Compute quantitative metrics
total_blocks = len(df)

# Count each category
runnable_y = len(df[df['Runnable'] == 'Y'])
runnable_n = len(df[df['Runnable'] == 'N'])
correct_y = len(df[df['Correct_Implementation'] == 'Y'])
correct_n = len(df[df['Correct_Implementation'] == 'N'])
redundant_y = len(df[df['Redundant'] == 'Y'])
redundant_n = len(df[df['Redundant'] == 'N'])
irrelevant_y = len(df[df['Irrelevant'] == 'Y'])
irrelevant_n = len(df[df['Irrelevant'] == 'N'])

# Calculate percentages
runnable_pct = (runnable_y / total_blocks) * 100
incorrect_pct = (correct_n / total_blocks) * 100
redundant_pct = (redundant_y / total_blocks) * 100
irrelevant_pct = (irrelevant_y / total_blocks) * 100

# Blocks that failed Runnable or Correct-Implementation
failed_blocks = len(df[(df['Runnable'] == 'N') | (df['Correct_Implementation'] == 'N')])

# Blocks with identified corrections
# Cell 8 has incorrect einsum which is corrected in Cell 10
blocks_with_corrections = 1

if failed_blocks > 0:
    correction_rate_pct = (blocks_with_corrections / failed_blocks) * 100
else:
    correction_rate_pct = 100.0

print("=" * 60)
print("QUANTITATIVE METRICS")
print("=" * 60)
print(f"\nTotal blocks evaluated: {total_blocks}")
print(f"\nRunnable%:    {runnable_pct:.2f}% ({runnable_y}/{total_blocks})")
print(f"Incorrect%:   {incorrect_pct:.2f}% ({correct_n}/{total_blocks})")
print(f"Redundant%:   {redundant_pct:.2f}% ({redundant_y}/{total_blocks})")
print(f"Irrelevant%:  {irrelevant_pct:.2f}% ({irrelevant_y}/{total_blocks})")
print(f"\nCorrection-Rate%: {correction_rate_pct:.2f}% ({blocks_with_corrections}/{failed_blocks} failed blocks have identified fixes)")

QUANTITATIVE METRICS

Total blocks evaluated: 42

Runnable%:    100.00% (42/42)
Incorrect%:   2.38% (1/42)
Redundant%:   2.38% (1/42)
Irrelevant%:  0.00% (0/42)

Correction-Rate%: 100.00% (1/1 failed blocks have identified fixes)


## Binary Checklist Summary

In [5]:
# Binary Checklist Summary

# C1: All core analysis code is runnable
c1_pass = runnable_n == 0
c1_status = "PASS" if c1_pass else "FAIL"

# C2: All implementations are correct
c2_pass = correct_n == 0
c2_status = "PASS" if c2_pass else "FAIL"

# C3: No redundant code
c3_pass = redundant_y == 0
c3_status = "PASS" if c3_pass else "FAIL"

# C4: No irrelevant code
c4_pass = irrelevant_y == 0
c4_status = "PASS" if c4_pass else "FAIL"

checklist_data = [
    {"Checklist Item": "C1", "Condition": "All core analysis code is runnable", "Status": c1_status},
    {"Checklist Item": "C2", "Condition": "All implementations are correct", "Status": c2_status},
    {"Checklist Item": "C3", "Condition": "No redundant code", "Status": c3_status},
    {"Checklist Item": "C4", "Condition": "No irrelevant code", "Status": c4_status},
]

checklist_df = pd.DataFrame(checklist_data)
print("=" * 70)
print("BINARY CHECKLIST SUMMARY")
print("=" * 70)
print()
display(checklist_df)

BINARY CHECKLIST SUMMARY



,Checklist Item,Condition,Status
0,C1,All core analysis code is runnable,PASS
1,C2,All implementations are correct,FAIL
2,C3,No redundant code,FAIL
3,C4,No irrelevant code,PASS


## Checklist Rationale

### C1: All core analysis code is runnable - PASS
All 42 cells are syntactically valid and have no obvious runtime blockers. Required imports are available in scope, and there are no undefined variables or missing dependencies.

### C2: All implementations are correct - FAIL
Cell 8 contains an incorrect implementation of the transformer model. The einsum operations use conflicting index notation ('k' used for multiple dimensions). However, this is corrected in Cell 10, which provides the proper implementation that the rest of the analysis uses.

### C3: No redundant code - FAIL
Cell 4 re-imports libraries (`transformer_lens`, `nn`, `F`, `Dataset`, `DataLoader`, `tqdm`, `json`) that were already imported in Cell 3. This appears to be a correction cell after Cell 3's version check failed, but the imports themselves are duplicated.

### C4: No irrelevant code - PASS
All cells contribute to the project goal of analyzing the modular addition circuit. The workflow follows the plan phases: setup, training, Fourier analysis, attention analysis, MLP analysis, ablation studies, and documentation.

## Final Summary

In [6]:
# Final Summary
print("=" * 70)
print("CODE EVALUATION SUMMARY")
print("=" * 70)

print("\nQUANTITATIVE METRICS:")
print(f"   Runnable%:         {runnable_pct:.2f}%")
print(f"   Incorrect%:        {incorrect_pct:.2f}%")
print(f"   Redundant%:        {redundant_pct:.2f}%")
print(f"   Irrelevant%:       {irrelevant_pct:.2f}%")
print(f"   Correction-Rate%:  {correction_rate_pct:.2f}%")

print("\nBINARY CHECKLIST:")
print(f"   C1 (All Runnable):     {c1_status}")
print(f"   C2 (All Correct):      {c2_status}")
print(f"   C3 (No Redundant):     {c3_status}")
print(f"   C4 (No Irrelevant):    {c4_status}")

print("\nISSUES FOUND:")
if correct_n > 0:
    print(f"   - {correct_n} block(s) with incorrect implementation (Cell 8: einsum index errors, fixed in Cell 10)")
if redundant_y > 0:
    print(f"   - {redundant_y} block(s) with redundant code (Cell 4: duplicate imports)")

print("\nOVERALL ASSESSMENT:")
print("   The code is well-structured and follows the research plan.")
print("   All code is syntactically valid and runnable.")
print("   One implementation error exists (Cell 8) but is corrected (Cell 10).")
print("   Minor redundancy in imports (Cell 4).")
print("   All code is relevant to the project goal.")

CODE EVALUATION SUMMARY

QUANTITATIVE METRICS:
   Runnable%:         100.00%
   Incorrect%:        2.38%
   Redundant%:        2.38%
   Irrelevant%:       0.00%
   Correction-Rate%:  100.00%

BINARY CHECKLIST:
   C1 (All Runnable):     PASS
   C2 (All Correct):      FAIL
   C3 (No Redundant):     FAIL
   C4 (No Irrelevant):    PASS

ISSUES FOUND:
   - 1 block(s) with incorrect implementation (Cell 8: einsum index errors, fixed in Cell 10)
   - 1 block(s) with redundant code (Cell 4: duplicate imports)

OVERALL ASSESSMENT:
   The code is well-structured and follows the research plan.
   All code is syntactically valid and runnable.
   One implementation error exists (Cell 8) but is corrected (Cell 10).
   Minor redundancy in imports (Cell 4).
   All code is relevant to the project goal.
